In [4]:
import os
import pandas as pd
import numpy as np
import joblib 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# 1. Thiết lập đường dẫn
PATH_INPUT = "../data/03_processed/pm25_features_ready.csv"
DIR_MODELS = "../models/"
os.makedirs(DIR_MODELS, exist_ok=True)

try:
    df = pd.read_csv(PATH_INPUT)
    
    # Xác định Biến mục tiêu (y) và Đặc trưng (X)
    target_col = 'pm25'
    drop_cols = ['datetime', target_col]
    
    X = df.drop(columns=drop_cols)
    y = df[target_col]

    # 2. CHIA DỮ LIỆU THEO TRỤC THỜI GIAN (70% Train - 15% Val - 15% Test)
    n_total = len(df)
    idx_train_end = int(n_total * 0.70)
    idx_val_end = int(n_total * 0.85)

    X_train, y_train = X.iloc[:idx_train_end], y.iloc[:idx_train_end]
    X_val, y_val = X.iloc[idx_train_end:idx_val_end], y.iloc[idx_train_end:idx_val_end]
    X_test, y_test = X.iloc[idx_val_end:], y.iloc[idx_val_end:]
    
    print(f"Tổng số dòng: {n_total}")
    print(f"Tập Train: {len(X_train)} dòng - Tập Val: {len(X_val)} dòng - Tập Test: {len(X_test)} dòng")
    print("-" * 50)
    print("Chuẩn bị dữ liệu hoàn tất")

except Exception as e:
    print(f"Lỗi ở bước đọc dữ liệu: {e}")

Tổng số dòng: 19704
Tập Train: 13792 dòng - Tập Val: 2956 dòng - Tập Test: 2956 dòng
--------------------------------------------------
Chuẩn bị dữ liệu hoàn tất


In [2]:
# Hàm tính điểm chuẩn hóa
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []

# 3. MÔ HÌNH 1: Persistence Model (Dự báo ngây thơ)
y_pred_persist = X_test['pm25_lag1']
results.append(evaluate_model("1. Persistence (Baseline)", y_test, y_pred_persist))
print("Đã ghi nhận điểm của Persistence Model")

# 4. MÔ HÌNH 2: Linear Regression (Baseline Thống kê)
print("Đang train Linear Regression")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
results.append(evaluate_model("2. Linear Regression", y_test, y_pred_lr))
print("Đã ghi nhận điểm của Linear Regression")

Đã ghi nhận điểm của Persistence Model
Đang train Linear Regression
Đã ghi nhận điểm của Linear Regression


In [9]:
# 5. MÔ HÌNH 3: Random Forest (Mô hình Đề xuất - AI)
print("Đang train Random Forest")
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=8,
    random_state=42,
    min_samples_leaf=5,
    n_jobs=-1)
rf_model.fit(X_train, y_train)

# --- THI THỬ VÀ CHECK OVERFITTING (Trên tập Val 15%) ---
mae_train = mean_absolute_error(y_train, rf_model.predict(X_train))
mae_val = mean_absolute_error(y_val, rf_model.predict(X_val))
print(f" Check Overfit - MAE Train: {mae_train:.2f} | MAE Val: {mae_val:.2f}")

# --- THI THẬT (Trên tập Test 15% - Chốt điểm) ---
y_pred_rf = rf_model.predict(X_test)

# Đảm bảo list results không bị cộng dồn nếu chạy ô này nhiều lần
if len(results) >= 3:
    results.pop() # Xóa kết quả RF cũ nếu có
results.append(evaluate_model("3. Random Forest", y_test, y_pred_rf))

# 6. IN BẢNG BÁO CÁO THÀNH TÍCH
print("\n" + "="*50)
print("BẢNG KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print("="*50)

df_results = pd.DataFrame(results)
df_results['MAE'] = df_results['MAE'].round(3)
df_results['RMSE'] = df_results['RMSE'].round(3)
df_results['R2'] = df_results['R2'].round(3)
print(df_results.to_string(index=False))
print("="*50)

# 7. Lưu lại mô hình xuất sắc nhất
path_model = os.path.join(DIR_MODELS, "random_forest_pm25.pkl")
joblib.dump(rf_model, path_model)
print(f"\nĐã lưu mô hình Random Forest tại: {path_model}")

Đang train Random Forest
 Check Overfit - MAE Train: 1.80 | MAE Val: 2.47

BẢNG KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST
                    Model   MAE  RMSE    R2
1. Persistence (Baseline) 2.815 4.401 0.888
     2. Linear Regression 2.357 3.998 0.907
         3. Random Forest 2.147 3.647 0.923

Đã lưu mô hình Random Forest tại: ../models/random_forest_pm25.pkl
